In [2]:
from pathlib import Path
import json

import joblib
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Пути

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "logistic_regression"

EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DATA_DIR / "train.csv"
VALID_PATH = PROCESSED_DATA_DIR / "valid.csv"

print("Train path:", TRAIN_PATH)
print("Valid path:", VALID_PATH)
print("Experiment dir:", EXPERIMENT_DIR)

Train path: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\data\processed\train.csv
Valid path: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\data\processed\valid.csv
Experiment dir: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\logistic_regression


In [4]:
# Загрузка данных

train_data = pd.read_csv(TRAIN_PATH)
valid_data = pd.read_csv(VALID_PATH)

print("Train data:", train_data.shape)
print("Valid data:", valid_data.shape)

display(train_data.head())
display(valid_data.head())

Train data: (536, 9)
Valid data: (116, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,1,119.0,88.0,41.0,170.0,45.3,0.507,26,0
1,0,107.0,76.0,30.0,126.0,45.3,0.686,24,0
2,1,130.0,70.0,13.0,105.0,25.9,0.472,22,0
3,0,152.0,82.0,39.0,272.0,41.5,0.270,27,0
4,0,129.0,110.0,46.0,130.0,67.1,0.319,26,1


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,4,137,84.0,30.0,126.0,31.2,0.252,30,0
1,2,87,72.0,23.0,126.0,28.9,0.773,25,0
2,0,102,78.0,40.0,90.0,34.5,0.238,24,0
3,0,124,56.0,13.0,105.0,21.8,0.452,21,0
4,0,165,76.0,43.0,255.0,47.9,0.259,26,0


In [ ]:
# Разделение данных на признаки и целевую переменную

TARGET_COLUMN = "Outcome"

X_train = train_data.drop(columns=[TARGET_COLUMN])
y_train = train_data[TARGET_COLUMN]

X_valid = valid_data.drop(columns=[TARGET_COLUMN])
y_valid = valid_data[TARGET_COLUMN]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_valid:", X_valid.shape)

X_train: (536, 8)
y_train: (536,)
X_valid: (116, 8)


# LogisticRegression

In [6]:
logistic_regression_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                solver="liblinear",
                random_state=57
            )
        )
    ]
)

logistic_regression_model.fit(X_train, y_train)

print("LogisticRegression model trained successfully.")

LogisticRegression model trained successfully.


In [7]:
# Предсказания 

y_valid_pred = logistic_regression_model.predict(X_valid)

y_valid_pred[:10]

array([0, 0, 0, 0, 1, 0, 0, 0, 0, 0])

In [8]:
# Оценка качества модели

metrics = {
    "model_name": "LogisticRegression",
    "accuracy": accuracy_score(y_valid, y_valid_pred),
    "precision": precision_score(y_valid, y_valid_pred),
    "recall": recall_score(y_valid, y_valid_pred),
    "f1": f1_score(y_valid, y_valid_pred)
}

metrics_df = pd.DataFrame([metrics])

display(metrics_df)

,model_name,accuracy,precision,recall,f1
0,LogisticRegression,0.818966,0.777778,0.682927,0.727273


In [9]:
print(classification_report(y_valid, y_valid_pred))

              precision    recall  f1-score   support

           0       0.84      0.89      0.86        75
           1       0.78      0.68      0.73        41

    accuracy                           0.82       116
   macro avg       0.81      0.79      0.80       116
weighted avg       0.82      0.82      0.82       116



In [10]:
conf_matrix = confusion_matrix(y_valid, y_valid_pred)

conf_matrix_df = pd.DataFrame(
    conf_matrix,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

display(conf_matrix_df)

,Predicted 0,Predicted 1
Actual 0,67,8
Actual 1,13,28


In [11]:
# Сохраняем модель и метрики

MODEL_PATH = EXPERIMENT_DIR / "model.joblib"
METRICS_PATH = EXPERIMENT_DIR / "metrics.json"

joblib.dump(logistic_regression_model, MODEL_PATH)

with open(METRICS_PATH, "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=4)

print("Model saved to:", MODEL_PATH)
print("Metrics saved to:", METRICS_PATH)

Model saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\logistic_regression\model.joblib
Metrics saved to: c:\Users\Qekqq\Desktop\Devops\devops_hw_1\experiments\logistic_regression\metrics.json


## Вывод

В качестве baseline-модели была обучена `LogisticRegression`. Для корректной работы модели признаки предварительно масштабировались с помощью `StandardScaler`.

Модель была обучена на train-выборке и оценена на validation-выборке. Полученные метрики сохранены в `experiments/logistic_regression/metrics.json`, а обученная модель — в `experiments/logistic_regression/model.joblib`.